# 2.2_huim — HUIM-inspired Stage 1 relevance model and catalogue shortlist

**Status of this notebook: implementation prepared, not yet executed.** No metric, training curve,
export, persistence or reload result exists until the cells below have been run in Colab. Cells that
print results only report what actually ran in that session; a prepared check is not evidence that it passed.

Pipeline position: *eligible article catalogue → **Stage 1 personalised shortlist (≤100)** → Stage 2
business reranker → final top 5*. This notebook implements Stage 1 only (no reranker, no ensemble).

Run order: `smoke` first (minutes, labelled smoke), then `development`, then `full` only after reading the
preflight storage/runtime estimates. Load-only: set `TRAIN_MODEL = False` and `LOAD_RUN_DIR`.

## 1. Business hypothesis and experiment overview

| Item | Value |
| --- | --- |
| Experiment ID | `2.2_huim` (neural track) |
| Owner | set `OWNER` below |
| Model | Compact HUIM-inspired click model: shared article tower, **candidate-conditioned attention over fixed-history items** (no recurrence), separate request/pair feature branch, concatenation or gated fusion, MLP head, BCE on logits |
| Business objective (Interim Report §1.1, §1.2.3) | Meaningful engagement and reader retention via relevant recommendations; Stage 1 protects **personal relevance through shortlist membership**; Stage 2 applies continuity, freshness and diversity within the shortlist |
| Hypothesis | Readers have several interests; weighting each past click by its similarity to the *candidate* (rather than averaging the whole history) should rank relevant articles higher on recorded slates and recover more known clicks in the catalogue top 100 |
| Evidence that would support it | Higher validation catalogue **known-click Recall@100 (primary)** and observed-click nDCG@100 (secondary) than the **uniform-pooling control** at the same budget and on the same fixed request sample; attention not collapsing to uniform. Recorded-slate AUC is a diagnostic only |
| Focused checks | (a) candidate attention vs uniform pooling; (b) concatenation vs gated fusion; (c) robustness when text/image vectors are hidden (most of the test catalogue has none) |

**Distinction from the parallel LSTM (2.1).** This model has no recurrence or positional encoding: the history is a
*set* of events, each carrying its own article, read-time bin, scroll bin and recency bin. Permuting aligned events
cannot change a score (unit-tested), while recency still influences attention because it stays attached to each event.

**What is adapted from HUIM (Xue et al.; BlackPearl code at commit `0d75efb`) and what is not**

| Aspect | HUIM paper | Released code | This adaptation |
| --- | --- | --- | --- |
| History pooling | DIN target attention, MLP([e_j, c, e_j−c, e_j⊙c]), softmax removed | FuxiCTR `DIN_Attention`, `din_use_softmax: false`, MLP [512,256] | Same input form, MLP [32,16]; **stable masked softmax by default**; the unnormalised masked sum (+LayerNorm, our addition) is an optional comparison; uniform pooling is a control |
| Item representation | category/subcat/sentiment/type embeddings + PCA-64 frozen text & image vectors (two attention groups) | same; PCA fitted on whole catalogue | One shared article tower: metadata embeddings + *separate* trained linear projections of frozen text (768-d) and image (1024-d) vectors, with missing-vector flags; no PCA; one attention group |
| History events | read-time and scroll bins | same, last 200 items | read-time/scroll quantile bins fitted on train history + fixed recency bins; last 50 fixed-history items |
| "Instant interest" | binned popularity/session/impression-level features (exposure counts, slate size, in-slate ranks…) | ~300 such features | **Not implemented**: these depend on the recorded slate, other users' test-period logs, or future totals. Nothing here is called "instant interest" |
| Fusion | BN(concat) → MLP, or gated (Eq. 11) | plain MLP (`CusDIN`); gated variant commented out | LayerNorm(concat[match, features]) → MLP; optional sigmoid gate (Eq. 11 style) |
| Loss | per-impression shifted logloss (stated not used for HUIM) | BCE | BCE on logits |
| History augmentation | val/test histories extended with later behaviour clicks | same | **No augmentation**: shared fixed histories only, events strictly before each split's first request |
| Leakage | paper reports 0.873 AUC with leaky features, 0.822 without | — | Only request-time-available inputs (audit table in §4) |

Course links: embeddings (metadata/article), attention (target attention), feature interactions
([h, c, h−c, h⊙c], preference crosses), dense networks (MLP head), regularisation (dropout, weight decay, modality
dropout, early stopping). The HUIM-style fusion as a whole is our adaptation, not an architecture taught in full.

**Metric contract (Experiments/README.md, commit 2869a48).** Stage 1 **primary**: mean known-click **Recall@100** over
validation requests, each scored against its **full time-eligible catalogue** — this selects checkpoints, early
stopping and tuning trials. **Secondary**: observed-click **nDCG@100** on the same shortlists (ideal DCG over all
original known clicks, so missed/ineligible clicks count against it). Ties: higher Recall@100, then nDCG@100, then the
earlier checkpoint/trial. **Diagnostic only**: recorded-slate mean impression AUC. nDCG@5 belongs to Stage 2's final
list. Training stays BCE on the recorded shown/clicked pairs; zero observed retrieval gain for unshown articles is
bookkeeping, never a negative label (no catalogue AUC/BCE).

**Evaluation pools (kept separate everywhere).** *Catalogue*: every time-eligible article per request (selection and
exports). *Recorded slates*: the logged shown candidates (diagnostic AUC, recorded-slate score files). Validation is
used repeatedly for selection, so its scores are **development results**; no reserved labelled final partition is
declared by the shared protocol. **Test is never a tuning or selection signal** (and has no labels).

## 2. Setup and run configuration

In [ ]:
# ---- Run configuration (edit here only) -------------------------------------------------
from pathlib import Path

RUN_MODE = "smoke"        # "smoke" (labelled, tiny) | "development" (aligned with 3.1: 5000/100/25) | "full"
TRAIN_MODEL = True        # False: load a saved bundle from LOAD_RUN_DIR and re-export without training
RESUME = True             # resume training/exports from the run's recovery state if present
SEED = 42
OWNER = "Winston Ho Rui Wei"

# Inputs (may be read-only). Use a shortcut path if the shared folder is not under MyDrive.
DATA_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder")
PATH_OVERRIDES = {}       # e.g. {"train_val_text_embeddings": "embeddings/train_val/text_embeddings.parquet"}

# Writable persistent storage for run bundles (separate from DATA_ROOT).
OUTPUT_ROOT = Path("/content/drive/MyDrive/BT4222 Experiment Runs/2.2_huim")
LOCAL_ROOT = Path("/content/huim_runs")    # fast session-local working copy. Export resume state lives here:
                                           # for multi-hour full exports set a Drive folder to survive VM loss.
RUN_ID = None             # None -> new "YYYYMMDD_HHMM_seed42_<mode>"; set an existing ID to resume it
LOAD_RUN_DIR = None       # TRAIN_MODEL=False: persistent run folder to load, e.g. OUTPUT_ROOT / "<run_id>"

# Code (private fork; authenticated with a Colab secret holding a GitHub token)
REPO_URL = "https://github.com/KoalaLyfestyle/BT4222-Final-Project.git"
REPO_BRANCH = "claude/focused-shannon-s4lmma"   # switch to "main" once merged
EXPECTED_COMMIT = None    # optional full SHA to pin the helper code
REPO_DIR = Path("/content/BT4222-Final-Project")
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
ALLOW_DIRTY_CHECKOUT = False

# Optional, off by default (each costs extra training time)
RUN_INTERRUPTION_CHECK = False   # bounded resume check (~2 x 40 steps on a training subset)
RUN_CONTROLS = False             # uniform-pooling / gated-fusion / unnormalised comparisons on the same selection sample
RUN_TUNING = False               # bounded Optuna search on validation catalogue Recall@100 (installs optuna)
TUNING_TRIALS, TUNING_MAX_EPOCHS, TUNING_TRAIN_REQUESTS = 8, 2, 2_000

# Shared pool + request scopes (Experiments/README.md, aligned with 3.1):
#  * pool published_at_or_before_request_v1 for EVERY split: ordered union train_val -> test articles,
#    first record wins (incl. missing fields); eligible iff known published_time <= impression_time.
#  * requests (never articles) are sampled: BLAKE2b-64(f"42:{split}:{behaviour_id}") prefix, processed by time.
#    RUN_MODE presets set three SEPARATE limits (training / selection_validation / export per split):
#      smoke 1000 / 25 / 5,  development 5000 / 100 / 25 (= 3.1),  full None / 100 / None (all requests).
#    Every selected inference request is scored against its FULL eligible pool.
REFERENCE_31_RUN = None   # optional: a 3.1 run folder (input_audit.json, eligibility/*.csv) whose saved
                          # request IDs are reused; our own hash selection is recorded and compared either way
INPUT_VERSION = None      # shared preprocessing release label, if the group has one
ENCODER_IDENTITY = {"text": "gemini-embedding-2, 768-d (Preprocessing/embedding_pipeline.py; confirm with its owner)",
                    "image": "EB-NeRD supplied image vectors, 1024-d"}
CONFIG_OVERRIDES = {}     # e.g. {"max_epochs": 3} ; full scope also needs {"max_export_pairs": <after budget>}

In [ ]:
# ---- Mount Drive and probe storage -------------------------------------------------------
import os, time, uuid
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab: set DATA_ROOT/OUTPUT_ROOT to local paths.")

assert DATA_ROOT.is_dir(), f"DATA_ROOT not found: {DATA_ROOT}"
for sub in ("feature_engineering", "train_val", "test"):
    assert (DATA_ROOT / sub).is_dir(), f"{DATA_ROOT} lacks {sub}/ — is this the shared data folder?"

def write_probe(root: Path) -> dict:
    """Create, read back and delete a small file. The shared input folder is never probed."""
    root.mkdir(parents=True, exist_ok=True)
    probe = root / f".write_probe_{uuid.uuid4().hex}"
    payload = os.urandom(1024)
    probe.write_bytes(payload)
    ok = probe.read_bytes() == payload
    probe.unlink()
    return {"root": str(root), "writable": ok}

STORAGE_PROBES = {"output_root": write_probe(OUTPUT_ROOT), "local_root": write_probe(LOCAL_ROOT)}
print(STORAGE_PROBES)
assert all(p["writable"] for p in STORAGE_PROBES.values())

In [ ]:
# ---- Dependency check BEFORE importing helper modules (no blanket upgrades) -------------
import importlib, subprocess, sys
REQUIRED = {"numpy": "1.24", "pandas": "2.0", "pyarrow": "14.0", "torch": "2.3", "matplotlib": "3.7"}
def vtuple(v): return tuple(int(x) for x in v.split("+")[0].split(".")[:2] if x.isdigit())
problems = {}
for name, minimum in REQUIRED.items():
    try:
        mod = importlib.import_module(name)
        if vtuple(mod.__version__) < vtuple(minimum):
            problems[name] = f"{mod.__version__} < {minimum}"
        print(f"{name:10s} {mod.__version__}")
    except ImportError as error:
        problems[name] = f"missing ({error})"
if RUN_TUNING:
    try:
        import optuna
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna>=3.6,<5"], check=True)
        import optuna
    print("optuna", optuna.__version__)
assert not problems, f"Dependency problems (install only these packages): {problems}"

In [ ]:
# ---- Code checkout: authenticated fetch of the private fork, revision recorded --------------
import base64, shutil
def _token():
    try:
        from google.colab import userdata
        return userdata.get(GITHUB_TOKEN_SECRET)
    except Exception:
        return os.environ.get(GITHUB_TOKEN_SECRET)
TOKEN = _token()

def git(*args, cwd=REPO_DIR, auth=False, check=True):
    cmd = ["git"]
    if auth and TOKEN:  # header only for this call; the token is not written to .git/config or printed
        basic = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
        cmd += ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}"]
    result = subprocess.run(cmd + list(args), cwd=cwd, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed: {result.stderr.replace(TOKEN or '<none>', '***')[:500]}")
    return result.stdout.strip()

if not (REPO_DIR / ".git").is_dir():
    git("clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR), cwd="/content", auth=True)
else:
    # Existing clone: never trust it blindly — fetch, refuse local edits, fast-forward.
    git("fetch", "origin", REPO_BRANCH, auth=True)
    dirty = git("status", "--porcelain")
    if dirty and not ALLOW_DIRTY_CHECKOUT:
        raise RuntimeError(f"Existing clone has local changes:\n{dirty}")
    git("checkout", REPO_BRANCH)
    git("merge", "--ff-only", f"origin/{REPO_BRANCH}")
if EXPECTED_COMMIT:
    git("checkout", EXPECTED_COMMIT)
CODE_REVISION = {"commit": git("rev-parse", "HEAD"), "branch": REPO_BRANCH,
                 "uncommitted_changes": bool(git("status", "--porcelain")),
                 "remote_head": git("rev-parse", f"origin/{REPO_BRANCH}", check=False)}
if EXPECTED_COMMIT:
    assert CODE_REVISION["commit"] == EXPECTED_COMMIT
print(CODE_REVISION)

In [ ]:
# ---- Import helpers from THIS checkout and verify consistency ------------------------------
import hashlib
EXP_DIR = REPO_DIR / "Experiments" / "2.2_huim"
for name in [m for m in sys.modules if m == "huim" or m.startswith("huim.")]:
    del sys.modules[name]          # drop stale modules from an earlier run of this kernel
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))
import huim
HELPER_HASHES = {}
for name in huim.HELPER_MODULES:
    module = importlib.import_module(f"huim.{name}")
    path = Path(module.__file__).resolve()
    assert path == (EXP_DIR / "huim" / f"{name}.py").resolve(), f"huim.{name} imported from {path}"
    HELPER_HASHES[name] = hashlib.sha256(path.read_bytes()).hexdigest()
tracked_changes = git("status", "--porcelain", "--", "Experiments/2.2_huim")
assert not tracked_changes or ALLOW_DIRTY_CHECKOUT, f"helper files differ from the commit:\n{tracked_changes}"

import json, numpy as np, pandas as pd, torch
from huim import pipeline as P, catalogue as C, features as F, data as D
from huim.io_utils import RunLogger, atomic_write_json, environment_info, read_json, dumps
from huim.model import parameter_counts

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("WARNING: no GPU; training and catalogue scoring will be very slow.")
SPLITS = D.SPLITS

CFG = P.make_config(RUN_MODE, seed=SEED, reference_31_run=str(REFERENCE_31_RUN) if REFERENCE_31_RUN else None,
                    **CONFIG_OVERRIDES)
LOADED_POOL_POLICY = None
if not TRAIN_MODEL:
    assert LOAD_RUN_DIR is not None and P.run_is_complete(Path(LOAD_RUN_DIR)), \
        "LOAD_RUN_DIR must be a persisted run with a valid completion marker (written after its reload check passed)"
    saved_cfg = read_json(Path(LOAD_RUN_DIR) / "run_config.json")["config"]
    for key in ("history_len", "d_model", "attention_mode", "attention_hidden", "fusion", "feature_hidden",
                "head_hidden", "text_dim", "image_dim"):
        CFG[key] = saved_cfg[key]   # model reconstruction settings come from the saved bundle
    # The evaluation pool/sampling are the CURRENT shared policy; the bundle's own selection provenance is kept.
    LOADED_POOL_POLICY = {"catalogue_policy": saved_cfg.get("catalogue_policy"),
                          "selection_protocol": read_json(Path(LOAD_RUN_DIR) / "model" / "model_config.json")
                          .get("selection", {}).get("version")}
RUN_ID = RUN_ID or f"{time.strftime('%Y%m%d_%H%M')}_seed{SEED}_{RUN_MODE}" + ("" if TRAIN_MODEL else "_reload")
RUN_DIR = LOCAL_ROOT / RUN_ID
PERSIST_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
log = RunLogger(RUN_DIR)
log(f"run {RUN_ID} | mode {RUN_MODE} | device {DEVICE} | commit {CODE_REVISION['commit'][:10]}")
if P.run_is_complete(PERSIST_DIR):
    log(f"NOTE: {PERSIST_DIR} already has a valid completion marker; choose a new RUN_ID to rerun.")
print(json.dumps({k: v for k, v in CFG.items()}, indent=1, default=str)[:3000])

## 3. Load and check shared data

Loads behaviours (`train`, `validation`, `test`), fixed histories, the **ordered union article catalogue**
(`train_val` then `test`, first record wins; the same pool for every split), both vector sources (same order, first
source record wins even when its vector is missing), and **only the keys and labels** of the shared exploded feature
tables (training pairs = recorded slates; no re-explosion, no catalogue negatives). Then the three request scopes are
fixed (training, selection-validation, export) by the shared hash rule and saved with the pool fingerprint. Checks: composite-key uniqueness, label agreement with
`article_ids_clicked`, history alignment and the temporal cutoff, vector decoding (dimensions, finite values, flags,
overlap agreement), coverage, exact eligible-pair counts and storage estimates, and agreement of our request-time
features with the shared Step 3 features on the recorded pairs.

In [ ]:
# Independent preflight of the ORIGINAL vector files (text and image, train_val and test): which reader
# decoded each file, its raw return type, dimensions, nulls and finite values. A file no reader can decode
# stops the run here; it is never treated as missing vectors or replaced by zeros.
with log.stage("vector file preflight"):
    VECTOR_FILES = D.vector_file_check(D.resolve_paths(DATA_ROOT, PATH_OVERRIDES),
                                       {"text": CFG["text_dim"], "image": CFG["image_dim"]})
atomic_write_json(RUN_DIR / "vector_file_check.json", VECTOR_FILES)
display(pd.DataFrame(VECTOR_FILES).T[["reader", "raw_return_type", "source_vector_type", "rows", "dimension",
                                       "usable", "null_vectors", "wrong_length", "non_finite", "passed"]])
assert all(v.get("passed") for v in VECTOR_FILES.values() if v.get("status") != "file_not_found"), VECTOR_FILES

In [ ]:
with log.stage("prepare inputs"):
    PRE_LOADED = None
    if not TRAIN_MODEL:
        PRE_LOADED = F.Preprocessor.from_json(read_json(Path(LOAD_RUN_DIR) / "model" / "preprocessing.json"))
    ws = P.prepare_inputs(CFG, DATA_ROOT, pre=PRE_LOADED, path_overrides=PATH_OVERRIDES, log=log)
    INPUT_MANIFEST = D.input_manifest(ws.paths)
atomic_write_json(RUN_DIR / "input_manifest.json", INPUT_MANIFEST)
summary = pd.DataFrame({s: {"requests": len(ws.behaviours[s]), "users": ws.behaviours[s].user_id.nunique(),
                            "recorded_pairs": len(ws.pairs[s]),
                            "clicked_pairs": int(np.nansum(ws.pairs[s].clicked)) if s != "test" else "unknown (no labels)",
                            "first_request": ws.behaviours[s].impression_time.min(),
                            "last_request": ws.behaviours[s].impression_time.max(),
                            "history_events_kept": ws.reports["history"][s]["events_kept"],
                            "catalogue_articles": ws.ctx[s].report["catalogue_articles"]} for s in SPLITS})
display(summary)
print(dumps(ws.reports["catalogues"]))

In [ ]:
with log.stage("request scopes + preflight"):
    SCOPES = P.resolve_request_scopes(ws, CFG)
    REQUEST_SCOPES = {"pool_fingerprint": ws.reports["pool_fingerprint"], "pool_policy": ws.reports["pool_policy"],
                      **P.scope_ids(ws, SCOPES)}
    atomic_write_json(RUN_DIR / "selection" / "request_scopes.json", REQUEST_SCOPES)
    PREFLIGHT = P.preflight(ws, CFG, [LOCAL_ROOT, OUTPUT_ROOT], SCOPES)
atomic_write_json(RUN_DIR / "preflight.json", PREFLIGHT)
print("Pool:", ws.reports["pool_fingerprint"]["policy"], ws.reports["pool_fingerprint"]["articles"], "articles;",
      "fingerprint", ws.reports["pool_fingerprint"]["ids_and_published_sha256"][:16])
print("Article lookup coverage:", ws.reports["article_lookup_coverage"])
display(pd.DataFrame({k: {c: v.get(c) for c in ("split", "limit", "requests", "population_requests", "used",
                                                "behaviour_ids_sha256")} | {"matches_31": (v.get("reference") or {})
                                                .get("matches_own_hash_selection")}
                      for k, v in PREFLIGHT["request_scopes"].items()}).T)
B = PREFLIGHT["budget"]
display(pd.DataFrame({s: {"planned_export_requests": B[s]["planned_export"]["requests"],
                          "planned_export_pairs": B[s]["planned_export"]["eligible_pairs"],
                          "planned_csv_GB": B[s]["planned_export"]["full_scores_csv_bytes_estimate"] / 1e9,
                          "all_requests": B[s]["all_requests"]["requests"],
                          "all_requests_pairs": B[s]["all_requests"]["eligible_pairs"],
                          "all_requests_csv_GB": B[s]["all_requests"]["full_scores_csv_bytes_estimate"] / 1e9,
                          "median_pool": B[s]["all_requests"]["eligible_per_request_median"],
                          "empty_pools_all": B[s]["all_requests"]["empty_pools"]} for s in SPLITS}).T)
print(f"selection: {B['selection_per_evaluation_pairs']:,} pairs per evaluation; planned export "
      f"{B['planned_export_pairs_total']:,} pairs (max_export_pairs {B['max_export_pairs']:,}); all requests "
      f"{B['all_requests_pairs_total']:,} pairs (formula bytes/row; refined after the throughput benchmark)")
display(pd.DataFrame(PREFLIGHT["coverage"]).T)
print("Recorded candidates not eligible in the catalogue:", PREFLIGHT["recorded_ineligible"])
print("Free bytes:", PREFLIGHT["disk_free_bytes"])
print("Consistency with shared Step 3 features:")
print(dumps(PREFLIGHT["consistency_with_shared_features"]))
for kind in ("text", "image"):
    for source, rep in PREFLIGHT["vector_decoding"][kind].items():
        if isinstance(rep, dict):
            print(kind, source, {k: rep.get(k) for k in ("rows", "dimension", "usable", "null_vectors",
                                                        "rows_overridden_by_earlier_source",
                                                        "later_vectors_not_backfilled_for_earlier_missing",
                                                        "overlap_max_abs_diff")})
    print(kind, "precedence:", PREFLIGHT["vector_decoding"][kind]["precedence"])

## 4. Prepare model inputs

All inputs are rebuilt from source tables for *any* (request, article) pair, so training, recorded-slate scoring and
catalogue scoring share one feature path. Preprocessing is fitted on train only: Step 3 vocabularies (reused),
subcategory/sentiment vocabularies and body-length scaling (train-referenced articles), engagement quantile bins
(train history events), dense standardisation (train requests and train recorded pairs). Flags stay 0/1.

In [ ]:
audit = pd.DataFrame(F.FEATURE_AUDIT, columns=["feature", "scope", "status", "reason"])
with pd.option_context("display.max_colwidth", 200):
    display(audit)
if TRAIN_MODEL and "request_scaler" not in ws.pre.state:
    with log.stage("fit dense scalers on train"):
        F.fit_dense_scalers(ws.pre, ws.ctx["train"], ws.art)
FEATURE_ORDER = {"request_dense": F.REQUEST_DENSE_COLUMNS(ws.pre), "pair_dense": F.PAIR_DENSE_COLUMNS,
                 "article_inputs": ["category", "article_type", "sentiment_label", "subcategory (bag)", "topics (bag)",
                                    "premium", "premium_missing", "body_z", "body_missing", "topic_count_z",
                                    "has_text", "has_image", "known_article", "text_vector(768, frozen)",
                                    "image_vector(1024, frozen)"],
                 "history_event_inputs": ["article (shared tower)", "read_time_bin", "scroll_bin", "recency_bin"]}
MISSING_POLICY = {
    "article_not_in_lookup": "index 0 -> learned unknown-article representation (known=0)",
    "missing_text_or_image_vector": "projection contributes 0; has_text/has_image=0; article kept",
    "missing_published_time": "age feature 0 + article_age_missing=1; never catalogue-eligible",
    "future_published_recorded_candidate": "age clipped at 0 (training/diagnostics only; ineligible in catalogue)",
    "missing_premium/subscriber": "access_missing=1",
    "missing_read_time/scroll": "bin 0 (missing) per event; request aggregates have *_missing flags",
    "empty_history": "masked attention returns a learned empty-history vector; history_empty=1",
    "unknown_category/type/topic": "embedding 'unknown' index; cross preferences use all catalogue values",
}
mcfg_preview = P.model_config(CFG, ws.pre, ws.art)
print(json.dumps(mcfg_preview.to_json(), indent=1))
print({s: {k: tuple(v.shape) for k, v in ws.ctx[s].tensors.items() if k.startswith(("hist_", "req_dense"))} for s in SPLITS})

## 5. Define and train / load model

Training: BCE on logits over the supplied recorded pairs; AdamW (no decay on biases/norms/embeddings), gradient
clipping, mixed precision on GPU. **Selection**: every `selection_interval_steps` optimiser steps and at each epoch end,
the fixed validation selection sample is scored against the full eligible catalogue and **Recall@100** (then nDCG@100,
then earlier) picks the checkpoint; `ReduceLROnPlateau` and early stopping (patience counted in selection evaluations,
`min_improvement`) follow the same objective. Recorded-slate AUC is logged at each evaluation as a diagnostic.
The selected checkpoint — not the last one — is restored and re-evaluated. Defaults are hypotheses, not tuned optima.

Recovery: `recovery/last.pt` holds model, optimiser, scheduler, GradScaler, progress, RNG states (CPU RNG restored as a
CPU tensor), the selected weights and the **selection protocol** (sample hash, interval, pool). A recovery file from a
different protocol — e.g. an earlier AUC-selected run — is refused, not reinterpreted: use a new `RUN_ID`.

In [ ]:
from huim.selection import SELECTION_PROTOCOL_VERSION
# One fixed selection sample shared by the main run, controls and tuning trials.
SELECTOR = P.make_selector(ws, CFG, DEVICE, SCOPES)
print(f"selection sample: {SELECTOR.protocol['sample']['requests']} of "
      f"{SELECTOR.protocol['sample']['of_validation_requests']} validation requests, full eligible catalogue each; "
      f"interval {CFG['selection_interval_steps']} steps + epoch ends")
SELECTION_PROVENANCE = "selected by validation catalogue Recall@100 in this run"
if TRAIN_MODEL:
    with log.stage("train / resume"):
        model, MODEL_CFG, TRAIN_STATE, TRAIN_SUBSET, SELECTION = P.train_model(
            ws, CFG, RUN_DIR, PERSIST_DIR, DEVICE, resume=RESUME, log=log, selector=SELECTOR)
    LEARNING = pd.DataFrame(TRAIN_STATE["history"])
    display(LEARNING)
    if not SELECTION["restored_matches_recorded"]:  # recorded in metadata; investigate before trusting the run
        log(f"WARNING: restored checkpoint re-evaluation {SELECTION['restored_checkpoint_reevaluation']} differs "
            f"from its recorded selection metrics")
else:
    from huim.persist import load_model
    model, MODEL_CFG, _pre, _meta = load_model(Path(LOAD_RUN_DIR), DEVICE)
    assert _pre.to_json() == ws.pre.to_json()
    SELECTION, TRAIN_STATE, TRAIN_SUBSET = _meta["selection"], None, {"rule": "loaded; not retrained"}
    LEARNING = pd.DataFrame(read_json(Path(LOAD_RUN_DIR) / "training_history.json"))
    if SELECTION.get("version") != SELECTION_PROTOCOL_VERSION:
        # Keep the original provenance; do not describe it as a Recall@100-selected shortlist model.
        SELECTION_PROVENANCE = (f"LEGACY selection ({SELECTION.get('version') or SELECTION.get('objective', 'unknown')}; "
                                f"pool {LOADED_POOL_POLICY['catalogue_policy']}); evaluated here on the current union "
                                "pool, but NOT selected under it — diagnostic only")
        log("WARNING: " + SELECTION_PROVENANCE)
    else:
        SELECTION_PROVENANCE = "loaded; originally selected by validation catalogue Recall@100"
    for rel in ("model/weights.pt", "model/model_config.json", "model/preprocessing.json", "model/article_ids.npy"):
        (RUN_DIR / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(Path(LOAD_RUN_DIR) / rel, RUN_DIR / rel)
    log(f"loaded model from {LOAD_RUN_DIR} (no fitting)")
PARAMS = parameter_counts(model)
print(SELECTION, PARAMS)

In [ ]:
# Optional bounded interruption/resume check (prepared; result only exists once executed).
INTERRUPTION_CHECK = None
if RUN_INTERRUPTION_CHECK and TRAIN_MODEL:
    from huim.train import Trainer, interruption_check, seed_everything
    from huim.model import build_model
    art_d, ctx_d = P.device_and_tensors(ws, DEVICE)
    # ~40k impressions (~110 steps at batch 4096) so the resume point (step 40) is genuinely mid-epoch
    _cfg = dict(CFG, train_requests=40_000)
    subset, _ = P.training_subset(ws, _cfg, P.resolve_request_scopes(ws, _cfg))
    def build_trainer(folder):  # optimisation-state resume only; selection is not exercised here
        seed_everything(SEED)
        m = build_model(P.model_config(CFG, ws.pre, ws.art), ws.pre).to(DEVICE)
        return Trainer(m, art_d, ctx_d["train"], ctx_d["validation"], DEVICE,
                       dict(CFG, train_pair_indices=subset, checkpoint_every_steps=0), folder, None, log,
                       selector=SELECTOR)
    with log.stage("interruption check"):
        INTERRUPTION_CHECK = interruption_check(build_trainer, RUN_DIR / "interruption_check", steps=40)
    atomic_write_json(RUN_DIR / "interruption_check.json", INTERRUPTION_CHECK)
    print(INTERRUPTION_CHECK)

In [ ]:
# Optional focused controls (same data, budget, seed, optimiser AND the same fixed catalogue selection sample).
CONTROLS = None
if RUN_CONTROLS and TRAIN_MODEL:
    def control_row(name, override, state, n_params):
        b = state["best"]
        return {"variant": name, **override, "selected_eval": b["eval_index"], "selected_epoch": b["epoch"],
                "selected_global_step": b["global_step"],
                "known_click_recall@100 (selection sample)": b["known_click_recall@100"],
                "observed_click_ndcg@100 (selection sample)": b["observed_click_ndcg@100"],
                "diagnostic_recorded_slate_auc": b["diagnostic_recorded_slate_auc"], "trainable_params": n_params}
    rows = []
    for name, override in {"uniform_pooling": {"attention_mode": "uniform"},
                           "gated_fusion": {"fusion": "gated"},
                           "unnormalised_attention": {"attention_mode": "unnormalised"}}.items():
        cdir = RUN_DIR / "controls" / name
        with log.stage(f"control {name}"):
            cm, _, cstate, _, _ = P.train_model(ws, dict(CFG, **override), cdir, None, DEVICE, resume=RESUME,
                                                log=log, selector=SELECTOR)
        rows.append(control_row(name, override, cstate, parameter_counts(cm)["total_trainable"]))
        del cm
    rows.append(control_row("selected (main run)", {"attention_mode": CFG["attention_mode"], "fusion": CFG["fusion"]},
                            TRAIN_STATE, PARAMS["total_trainable"]))
    CONTROLS = pd.DataFrame(rows)
    CONTROLS.to_csv(RUN_DIR / "controls" / "controls.csv", index=False)
    display(CONTROLS)

In [ ]:
# Optional bounded Optuna search on validation catalogue Recall@100 (same fixed selection sample; development
# data; test is never evaluated). Does not replace the main run automatically: copy the chosen values into
# CONFIG_OVERRIDES and rerun, recording why.
STUDY_SUMMARY = None
if RUN_TUNING and TRAIN_MODEL:
    import optuna
    from huim.tune import run_study, select_best_trial, SEARCH_SPACE
    def objective(trial, params):
        tcfg = dict(CFG, **params, max_epochs=TUNING_MAX_EPOCHS, train_requests=TUNING_TRAIN_REQUESTS,
                    checkpoint_every_steps=0)
        tdir = RUN_DIR / "tuning" / f"trial_{trial.number:03d}"
        def report(record):  # intermediate Recall@100 for pruning
            trial.report(record["known_click_recall@100"], record["eval_index"])
            if trial.should_prune():
                raise optuna.TrialPruned()
        _, _, tstate, _, _ = P.train_model(ws, tcfg, tdir, None, DEVICE, resume=True, log=log,
                                           selector=SELECTOR, on_evaluation=report)
        b = tstate["best"]
        for key in ("observed_click_ndcg@100", "diagnostic_recorded_slate_auc", "epoch", "global_step"):
            trial.set_user_attr(key, b[key])
        return b["known_click_recall@100"]
    with log.stage("optuna"):
        study = run_study(objective, f"huim_{RUN_ID}", RUN_DIR / "tuning", PERSIST_DIR, TUNING_TRIALS, SEED)
    STUDY_SUMMARY = {"search_space": SEARCH_SPACE, "n_trials": len(study.trials),
                     "states": pd.Series([t.state.name for t in study.trials]).value_counts().to_dict(),
                     "selected_trial": select_best_trial(study), "sampler": "TPE(seed)",
                     "pruner": "MedianPruner(n_startup_trials=3, n_warmup_steps=1) on intermediate Recall@100",
                     "objective": SELECTOR.protocol["objective"], "selection_sample": SELECTOR.protocol["sample"],
                     "budget": {"trials": TUNING_TRIALS, "epochs": TUNING_MAX_EPOCHS,
                                "train_requests": TUNING_TRAIN_REQUESTS}}
    atomic_write_json(RUN_DIR / "tuning" / "summary.json", STUDY_SUMMARY)
    display(study.trials_dataframe())

## 6. Validation diagnostics and explainability

**Primary** (selection sample, the selected and restored checkpoint): catalogue known-click Recall@100;
**secondary**: observed-click nDCG@100. Full-validation values for the selected model come from the §7 contract export
of every validation request (`development`/`full` modes) and are reported separately from the sampled values.
**Diagnostic**: recorded-slate AUC. Learning curves show training loss, Recall@100, nDCG@100 and diagnostic AUC per
selection evaluation. Focused checks: (i) Recall@100/nDCG@100 with all vectors hidden — most test-catalogue articles
have none; (ii) attention weights for clicked vs unclicked candidates and attention entropy relative to uniform pooling.
Attention is diagnostic, not proof of reader motives or causality.

In [ ]:
import matplotlib.pyplot as plt
with log.stage("validation diagnostics"):
    SELECTED_SAMPLE = SELECTOR.evaluate(model)   # the restored/loaded selected checkpoint
    VAL_DIAG_AUC, _ = P.evaluate_recorded(model, P.device_and_tensors(ws, DEVICE)[1]["validation"],
                                          P.device_and_tensors(ws, DEVICE)[0], DEVICE)
    VECTOR_MASKED = P.vector_masked_metrics(model, ws, DEVICE, selector=SELECTOR)
    ATTENTION = P.attention_examples(model, ws, DEVICE, n_requests=3, seed=SEED)
display(pd.DataFrame({
    "selected model (selection sample)": {k: SELECTED_SAMPLE[k] for k in (
        "known_click_recall@100", "observed_click_ndcg@100", "eligible_only_recall@100", "requests_evaluated",
        "requests_with_ties_crossing_cutoff", "empty_pool_requests_with_known_clicks")},
    "vectors hidden (selection sample)": VECTOR_MASKED}).T)
print("Diagnostic recorded-slate validation AUC:", VAL_DIAG_AUC["mean_impression_auc"],
      f"({VAL_DIAG_AUC['auc_eligible_impressions']} impressions; {VAL_DIAG_AUC['auc_excluded_single_class']} excluded)")
print("Selection provenance:", SELECTION_PROVENANCE)
if len(LEARNING) and "known_click_recall@100" in LEARNING:  # legacy (AUC-selected) histories are not plotted
    fig, axes = plt.subplots(1, 3, figsize=(13, 3))
    axes[0].plot(LEARNING.global_step, LEARNING.train_loss_since_last_eval, marker="o"); axes[0].set(title="train BCE (since last eval)", xlabel="step")
    axes[1].plot(LEARNING.global_step, LEARNING["known_click_recall@100"], marker="o", label="Recall@100 (primary)")
    axes[1].plot(LEARNING.global_step, LEARNING["observed_click_ndcg@100"], marker="s", label="nDCG@100 (secondary)")
    axes[1].set(title="validation catalogue (selection sample)", xlabel="step"); axes[1].legend(fontsize=7)
    axes[2].plot(LEARNING.global_step, LEARNING.diagnostic_recorded_slate_auc, marker="o", color="grey")
    axes[2].set(title="recorded-slate AUC (diagnostic)", xlabel="step")
    sel_step = SELECTION.get("selected_checkpoint", {}).get("global_step")
    for ax in axes[1:]:
        if sel_step is not None:
            ax.axvline(sel_step, ls="--", lw=0.8, color="k")
    fig.tight_layout(); fig.savefig(RUN_DIR / "diagnostics" / "learning_curves.png", dpi=120); plt.show()
print("mean attention entropy / log(n_valid):", ATTENTION["mean_entropy_ratio_vs_uniform"],
      "(1.0 = uniform; lower = more selective)")
ex = pd.DataFrame(ATTENTION["examples"])
display(ex.head(20))
atomic_write_json(RUN_DIR / "diagnostics" / "attention_examples.json", ATTENTION)
if len(ex):
    first = ex[ex.behaviour_id == ex.behaviour_id.iloc[0]]
    fig, ax = plt.subplots(figsize=(7, 3))
    for _, r in first.iterrows():
        ax.plot(range(1, len(r.top_weights) + 1), r.top_weights, marker="o",
                label=f"{r.candidate_article_id} ({'clicked' if r.clicked else 'not clicked'})")
    ax.set(xlabel="history item rank by weight", ylabel="attention weight", title=f"Request {first.behaviour_id.iloc[0]}")
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(RUN_DIR / "diagnostics" / "attention_example.png", dpi=120)
    plt.show()

## 7. Predict and export all splits

1. **Recorded-slate diagnostics** for all splits (`diagnostics/recorded_slates/{split}_scores.csv`), every original
   candidate exactly once, with catalogue-ineligible recorded candidates listed separately.
2. **Catalogue exports** to the contract paths for the export scope of each split (`export_requests`: 5 / 25 /
   all): every exported request is scored against its **full eligible union pool**; full scores + top-100, validated
   (all blocks for small exports). A bounded throughput benchmark refines the budget; `max_export_pairs` stops an
   infeasible export, and the storage gate decides whether full score CSVs fit — a split that fails keeps complete
   top-100 shortlists (re-validated against recomputed full scores) and the manifest says why. The pool never changes.
3. **Metric scopes stay separate**: `selection_validation` (the selection requests) and `exported_validation` (the
   exported validation requests). Compare only identical request IDs and pools.

In [ ]:
with log.stage("recorded-slate exports"):
    RECORDED, RECORDED_LOGITS = P.export_recorded(model, ws, RUN_DIR, DEVICE)
display(pd.DataFrame({s: {"rows": r["rows"], "valid": r["validation"]["passed"],
                          "ineligible_in_catalogue": r["catalogue_ineligible_recorded_candidates"],
                          **({"diagnostic_recorded_slate_auc": r["metrics"]["mean_impression_auc"]}
                             if "metrics" in r else {})} for s, r in RECORDED.items()}).T)

In [ ]:
EXPORT_SETS = {s: SCOPES[f"export_{s}"]["positions"] for s in SPLITS}
with log.stage("bounded throughput benchmark"):
    art_d, ctx_d = P.device_and_tensors(ws, DEVICE)
    THROUGHPUT = {s: C.measure_throughput(C.CatalogueScorer(model, ctx_d[s], art_d, DEVICE, CFG["pair_chunk"],
                                                            CFG["inference_amp"]),
                                          EXPORT_SETS[s][: 2 * CFG["request_block"]], CFG["request_block"], n_blocks=2)
                  for s in SPLITS}
tp = THROUGHPUT["test"]
BUDGET = P.export_budget(ws, CFG, SCOPES, bytes_per_row=tp.get("csv_bytes_per_row"),
                         pairs_per_second=tp.get("pairs_per_second_scoring_only"))
display(pd.DataFrame({s: {**{f"planned_{k}": v for k, v in BUDGET[s]["planned_export"].items()},
                          **{f"all_{k}": v for k, v in BUDGET[s]["all_requests"].items()}} for s in SPLITS}).T)
P.check_export_budget(BUDGET)     # stops (never shrinks the pool) if the planned export is too large
GATE = P.full_score_gate(BUDGET, CFG, [LOCAL_ROOT, OUTPUT_ROOT])
display(pd.DataFrame(GATE).T)
with log.stage("catalogue exports (export scope, full eligible pools)"):
    EXPORTS = P.export_catalogue(model, ws, CFG, RUN_DIR, DEVICE, EXPORT_SETS, {s: GATE[s]["write"] for s in SPLITS},
                                 RECORDED_LOGITS, log=log,
                                 check_blocks=None if BUDGET["planned_export_pairs_total"] <= 50_000_000
                                 else CFG["validation_check_blocks"])
EXPORTED_VALIDATION = EXPORTS["validation"]["summary"].get("recall", {})
display(pd.DataFrame({s: {**r["summary"].get("recall", {}), "requests": r["summary"]["requests"],
                          "scored_pairs": r["summary"]["scored_pairs"], "empty_pools": r["summary"]["empty_pools"],
                          "full_scores_written": r["summary"]["full_scores_written"],
                          "validation_passed": r["validation"]["passed"],
                          "recorded_vs_catalogue_max_diff": r["validation"]["checks"].get(
                              "recorded_vs_catalogue_max_abs_logit_diff")} for s, r in EXPORTS.items()}).T)
assert all(r["validation"]["passed"] for r in EXPORTS.values()), {s: r["validation"]["errors"] for s, r in EXPORTS.items()}

In [ ]:
# ---- Manifests, metrics, run configuration, dependencies; then persist (marker after the reload check) ----
def file_entry(path):
    return {"path": str(Path(path).relative_to(RUN_DIR)), "bytes": Path(path).stat().st_size} if Path(path).is_file() else None

POOL_FP = ws.reports["pool_fingerprint"]
# Sampled iff any ACTUAL scope is smaller than its population (not inferred from the configured limits)
SAMPLED = (any(SCOPES[f"export_{s}"]["info"]["requests"] < len(ws.ctx[s].requests) for s in SPLITS)
           or SCOPES["training"]["info"]["requests"] < len(ws.ctx["train"].requests))
contract = {}
for s in SPLITS:
    res = EXPORTS[s]
    contract[s] = {
        "rankings": file_entry(RUN_DIR / "rankings" / s / "submission_top_100.csv"),
        "catalogue_scores": file_entry(RUN_DIR / "predictions" / s / "submission_click_probabilities.csv"),
        "catalogue_scores_status": "written" if res["summary"]["full_scores_written"] else
                                   "not exported: " + (GATE[s]["reason"] or "unknown"),
        "request_manifest": file_entry(RUN_DIR / "diagnostics" / "catalogue" / f"{s}_requests.csv"),
        "recorded_slate_scores": file_entry(RUN_DIR / "diagnostics" / "recorded_slates" / f"{s}_scores.csv"),
        "export_scope": SCOPES[f"export_{s}"]["info"], "population_requests": int(len(ws.ctx[s].requests)),
        "requests_exported": res["summary"]["requests"], "scored_pairs": res["summary"]["scored_pairs"],
        "empty_pools": res["summary"]["empty_pools"], "validation": res["validation"],
    }
CATALOGUE_VERSIONS = {name: INPUT_MANIFEST.get(f"articles.{name}") for name in ("train_val", "test")}
EXPORT_MANIFEST = {
    "export_schema_version": C.EXPORT_SCHEMA_VERSION, "prediction_pool": C.PREDICTION_POOL,
    "pool_policy": ws.reports["pool_policy"], "pool_fingerprint": POOL_FP,
    "scope": {"run_mode": RUN_MODE, "sampled_development": SAMPLED,
              "limits": {"training": CFG["train_requests"], "selection_validation": CFG["selection_requests"],
                         "export": CFG["export_requests"]},
              "request_ids": "selection/request_scopes.json", "sampling_rule": SCOPES["training"]["info"]["rule"],
              "sampling_seed": CFG["request_sampling_seed"],
              "note": "a full-pool result on a request sample is not a full-split result"},
    "is_complete_submission": bool(not SAMPLED and all(contract[s]["catalogue_scores"] for s in SPLITS)
                                   and all(contract[s]["validation"]["passed"] for s in SPLITS)),
    "eligibility_rules": C.ELIGIBILITY_RULES, "catalogue_policy": CFG["catalogue_policy"],
    "catalogue_versions": CATALOGUE_VERSIONS, "shortlist_size": C.SHORTLIST_SIZE,
    "final_display_size": C.FINAL_DISPLAY_SIZE, "score_semantics": C.SCORE_SEMANTICS,
    "score_file_name": "submission_click_probabilities.csv",
    "tie_breaker": "descending score, then ascending original article_id",
    "behaviour_id": "f'{user_id}:{impression_id}' from original integer IDs",
    "contract_exports": contract,
    "recorded_slates": {s: {k: v for k, v in r.items() if k != "metrics"} for s, r in RECORDED.items()},
    "feasibility": {"preflight_budget": PREFLIGHT["budget"], "refined_budget": BUDGET, "throughput": THROUGHPUT,
                    "gate": GATE},
}
atomic_write_json(RUN_DIR / "export_manifest.json", EXPORT_MANIFEST)

def scope_meta(name):
    info = SCOPES[name]["info"]
    return {"purpose": name, "split": info["split"], "requests": info["requests"], "limit": info["limit"],
            "behaviour_ids_sha256": info["behaviour_ids_sha256"], "request_ids_file": "selection/request_scopes.json",
            "used": info["used"], "reference": info.get("reference"), "pool_policy": POOL_FP["policy"],
            "pool_fingerprint": POOL_FP["ids_and_published_sha256"],
            "model": {"run_id": RUN_ID, "selected_checkpoint": SELECTION.get("selected_checkpoint"),
                      "selection_provenance": SELECTION_PROVENANCE}}
METRICS = {
    "definitions": P.METRIC_DEFINITIONS,
    "selection_protocol": SELECTOR.protocol, "selection": SELECTION, "selection_provenance": SELECTION_PROVENANCE,
    # Two separate metric scopes (named as in 3.1). Never compare them as if they used the same requests.
    "selection_validation": {**{k: SELECTED_SAMPLE[k] for k in (
        "known_click_recall@100", "observed_click_ndcg@100", "eligible_only_recall@100", "requests_evaluated",
        "requests_without_known_clicks", "empty_pool_requests_with_known_clicks", "requests_with_ties_crossing_cutoff",
        "clicked_articles_total", "clicked_articles_eligible", "clicked_articles_in_shortlist", "scored_pairs")},
        "scope": scope_meta("selection_validation"), "source": "re-scored selected checkpoint (no CSV)"},
    "exported_validation": {**EXPORTED_VALIDATION, "scope": scope_meta("export_validation"),
                            "source": "validated exported top-100 shortlists; original known-click denominators"},
    "same_requests_in_both_scopes": SCOPES["selection_validation"]["info"]["behaviour_ids_sha256"]
                                    == SCOPES["export_validation"]["info"]["behaviour_ids_sha256"],
    "primary": "known_click_recall@100", "secondary": "observed_click_ndcg@100",
    "vectors_hidden_selection_validation": VECTOR_MASKED,
    "learning_curve": LEARNING.to_dict(orient="records"),
    "diagnostic_recorded_slates": {"validation": VAL_DIAG_AUC, "train_in_sample": RECORDED["train"]["metrics"],
                                   "note": "diagnostic only; not used for Stage 1 selection; nDCG@5 here is a "
                                           "recorded-slate diagnostic, the evaluated nDCG@5 belongs to Stage 2"},
    "catalogue_other": {"train_exported_in_sample": EXPORTS["train"]["summary"].get("recall"),
                        "note": "train metrics are in-sample diagnostics"},
    "test": "unavailable: official test split has no click labels; never used for tuning or selection",
    "attention_entropy_ratio": ATTENTION["mean_entropy_ratio_vs_uniform"],
    "controls": None if CONTROLS is None else CONTROLS.to_dict(orient="records"),
    "tuning": STUDY_SUMMARY, "interruption_check": INTERRUPTION_CHECK,
    "notes": ["Validation was used for checkpoint selection: development results, not an untouched test.",
              "selection_validation and exported_validation use different request sets unless their hashes match.",
              "Recall@100/nDCG@100 measure recovery of observed clicks, not unbiased catalogue relevance or CTR."],
}
atomic_write_json(RUN_DIR / "metrics.json", METRICS)
RUN_CONFIG = {
    "experiment_id": P.EXPERIMENT_ID, "run_id": RUN_ID, "owner": OWNER, "seed": SEED, "run_mode": RUN_MODE,
    "train_model": TRAIN_MODEL, "loaded_from": str(LOAD_RUN_DIR) if LOAD_RUN_DIR else None,
    "code": {**CODE_REVISION, "helper_sha256": HELPER_HASHES, "repo_url": REPO_URL},
    "environment": environment_info(), "config": CFG, "storage_probes": STORAGE_PROBES,
    "data_root": str(DATA_ROOT), "resolved_paths": ws.paths.as_dict(), "input_manifest": "input_manifest.json",
    "feature_case": CFG["feature_case"], "feature_order": FEATURE_ORDER, "feature_audit": F.FEATURE_AUDIT,
    "missing_value_policy": MISSING_POLICY, "excluded_inputs": {**D.EXCLUDED_ARTICLE_COLUMNS, **D.EXCLUDED_BEHAVIOUR_COLUMNS},
    "preprocessing": "model/preprocessing.json", "model_config": MODEL_CFG.to_json(), "parameters": PARAMS,
    "training_subset": TRAIN_SUBSET, "selection": SELECTION, "selection_protocol": SELECTOR.protocol,
    "selection_provenance": SELECTION_PROVENANCE,
    "selection_sample_file": "selection/selection_requests.json", "embedding_sources": {
        "text": "gemini-embedding-2, 768-d, frozen lookup (Preprocessing/embedding_pipeline.py)",
        "image": "EB-NeRD supplied, 1024-d, frozen lookup", "combination": f"first source wins: {CFG['vector_source_order']}",
        "metadata": {k: str(ws.paths.vector_metadata.get(k)) for k in ws.paths.vector_metadata},
        "encoder_identity_declared": ENCODER_IDENTITY,
        "encoder_identity_from_metadata": {k: {f: read_json(v).get(f) for f in ("text_model", "text_dimension",
                                                                                 "image_source", "image_dimension")}
                                           for k, v in ws.paths.vector_metadata.items() if v is not None},
        "precedence": ws.vector_reports["text"]["precedence"]},
    "input_version": INPUT_VERSION,
    "pool_policy": ws.reports["pool_policy"], "pool_fingerprint": ws.reports["pool_fingerprint"],
    "article_lookup_coverage": ws.reports["article_lookup_coverage"],
    "request_scopes": {k: v["info"] for k, v in SCOPES.items()}, "request_scope_ids": "selection/request_scopes.json",
    "candidate_counts": {s: {"requests": int(len(ws.ctx[s].requests)), "recorded_pairs": int(len(ws.pairs[s])),
                             "planned_export_pairs": BUDGET[s]["planned_export"]["eligible_pairs"],
                             "all_requests_pairs": BUDGET[s]["all_requests"]["eligible_pairs"]} for s in SPLITS},
    "loaded_bundle_pool_policy": LOADED_POOL_POLICY if not TRAIN_MODEL else None,
    "history": {"source": "shared fixed history.parquet per split", "cutoff": "events strictly before each split's first request",
                "augmentation": "none (later behaviour clicks are not appended)", "items": CFG["history_len"],
                "reports": ws.reports["history"]},
    "coverage": PREFLIGHT["coverage"], "export_schema_version": C.EXPORT_SCHEMA_VERSION,
    "prediction_pool": C.PREDICTION_POOL, "eligibility_rules": C.ELIGIBILITY_RULES,
    "catalogue_versions": CATALOGUE_VERSIONS, "shortlist_size": C.SHORTLIST_SIZE, "score_semantics": C.SCORE_SEMANTICS,
    "export_manifest": "export_manifest.json", "tuning_objective": SELECTOR.protocol["objective"] + "; ties: nDCG@100, then earlier trial",
    "training_objective": "BCE on logits over supplied recorded shown/clicked pairs",
    "vector_file_check": "vector_file_check.json",
    "persistent_location": str(PERSIST_DIR),
}
atomic_write_json(RUN_DIR / "run_config.json", RUN_CONFIG)
(RUN_DIR / "requirements.txt").write_text(P.requirements_text())

with log.stage("persist run bundle (copy + verify; marker comes after the reload check)"):
    PERSISTED = P.persist_run(RUN_DIR, PERSIST_DIR, log=log)
print("Persisted (not yet marked complete):", PERSIST_DIR)

## 8. Reload check

Loads weights, configuration and preprocessing **from the persistent copy** (no fitting), re-scores validation
recorded candidates and the first dev-sample catalogue requests, and compares with the persisted exports (tolerance
1e-4 on logits). Only if this passes is `_COMPLETE_RUN.json` written — last — so an unverified bundle is never treated
as complete. The result is only evidence once this cell has run.

In [ ]:
with log.stage("reload check from persistent copy"):
    RELOAD = P.reload_check(PERSIST_DIR, ws, CFG, DEVICE)
atomic_write_json(RUN_DIR / "reload_check.json", RELOAD)
shutil.copyfile(RUN_DIR / "reload_check.json", PERSIST_DIR / "reload_check.json")
print(json.dumps(RELOAD, indent=1))
assert RELOAD["passed"], "reload check failed — see reload_check.json; the run is NOT marked complete"
P.mark_run_complete(RUN_DIR, PERSIST_DIR, RELOAD, log=log)
print("complete:", P.run_is_complete(PERSIST_DIR))

## 9. Business interpretation and handoff

Fill this in **from the executed outputs above**; do not carry numbers from elsewhere.

* Validation (development) catalogue Recall@100 (primary) and observed-click nDCG@100 (secondary) — selection sample
  and, where run, every validation request — and, if run, the uniform-pooling and gated-fusion controls on the same
  sample: does candidate-conditioned matching help at a comparable budget? If not, the simpler control is sufficient.
  Recorded-slate AUC is a diagnostic; nDCG@5 is for Stage 2's final list.
* Eligibility exclusions of known clicks (they stay in the denominators), ties crossing the cutoff, and the
  shortlist's vector coverage / median age / category spread — inputs to Stage 2's freshness and diversity work.
* Vector-hidden robustness: how much does ranking degrade without text/image vectors (most test-catalogue articles)?
* Limitations: exposure bias (trained only on shown slates), unknown outcomes outside recorded slates, catalogue
  snapshot differences between `train_val` and `test`, offline clicks ≠ retention.

Handoff for the Stage 2 owner: the persisted bundle at `PERSIST_DIR` (rankings, catalogue scores where written,
recorded-slate diagnostics, request manifests, model bundle). The cell below assembles the small review bundle.

In [ ]:
# ---- Small review bundle (no full data, no model binaries) --------------------------------------
import zipfile
REVIEW = RUN_DIR / f"review_bundle_{RUN_ID}.zip"
small = ["run_config.json", "metrics.json", "export_manifest.json", "preflight.json", "input_manifest.json",
         "training_history.json", "reload_check.json", "interruption_check.json", "requirements.txt",
         "logs/run.log", "logs/errors.log", "controls/controls.csv", "tuning/trials.csv", "tuning/summary.json",
         "diagnostics/attention_examples.json", "diagnostics/attention_example.png",
         "selection/request_scopes.json", "selection/selection_requests.json", "vector_file_check.json",
         "diagnostics/learning_curves.png"]
small += [str(p.relative_to(RUN_DIR)) for p in RUN_DIR.glob("diagnostics/catalogue/*_export_progress.json")]
small += [str(p.relative_to(RUN_DIR)) for p in RUN_DIR.glob("diagnostics/catalogue/*_requests.csv")]
small += [str(p.relative_to(RUN_DIR)) for p in RUN_DIR.glob("diagnostics/recorded_slates/*_catalogue_ineligible.csv")]
with zipfile.ZipFile(REVIEW, "w", zipfile.ZIP_DEFLATED) as zf:
    for rel in small:
        if (RUN_DIR / rel).is_file() and (RUN_DIR / rel).stat().st_size < 200 * 1024 ** 2:
            zf.write(RUN_DIR / rel, rel)
shutil.copyfile(REVIEW, PERSIST_DIR / REVIEW.name)
print(f"Review bundle: {PERSIST_DIR / REVIEW.name} ({REVIEW.stat().st_size / 1e6:.1f} MB)")
print("Bundle location for Stage 2:", PERSIST_DIR)
print({s: contract[s]["catalogue_scores_status"] for s in SPLITS})
if IN_COLAB:
    from google.colab import files
    files.download(str(REVIEW))